# Knee MRI — EfficientNet-B0 2.5D MIL: walkthrough

This notebook drives the implementation in `src/knee_mri`. It imports the modules and calls
the same functions the CLI uses — the implementation is **not** duplicated here.

Order: schema and label checks → manifest and series selection → deterministic cache →
QC gallery → folds → the tiny checks → one fold of training.

Heavy steps (full manifest, full cache, full fold training) are marked and are easier to run
from a terminal, where they can be left alone:

```powershell
cd src
python -m knee_mri.cli build-manifest
python -m knee_mri.cli build-cache
```

## 0. Setup

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd

from knee_mri.config import load_config
from knee_mri.utils import log_environment, seed_everything

cfg = load_config()                      # src/config.yaml
seed_everything(int(cfg.seed))
env = log_environment()
print({k: env[k] for k in ("python", "torch", "torchvision", "pydicom", "sklearn", "cuda_available")})
print("data root:", cfg.paths.data_root)
print("work dir :", cfg.paths.work_dir)

In [ ]:
# Anything can be overridden without editing the YAML:
#   cfg = load_config(overrides=["data.image_size=288", "train.microbatch_studies=2"])
print(cfg.dumps()[:1200])

## 1. Input schema, label contract and coverage

`validate_inputs` checks identifier types, uniqueness, the many-to-one series→study join,
orphans in both directions, unexpected values and labelled coverage. Findings are typed
(`info` / `warning` / `error`); errors stop the pipeline.

In [ ]:
from knee_mri.schema import validate_inputs, read_id_csv
from knee_mri.constants import STUDY_ID, TARGETS

report = validate_inputs(cfg)
report.log()
print("\nerrors:", len(report.errors), "| warnings:", len(report.warnings))
pd.DataFrame([f.as_row() for f in report.findings])

In [ ]:
train_df = read_id_csv(cfg.paths.train_csv)
study_ids = sorted(set(train_df[STUDY_ID].dropna().astype(str)))
print(f"{len(study_ids)} studies in train.csv")
print("target order:", TARGETS)
train_df.head(3)

### Label table

Statuses become `(target, weight)` pairs. Unknown stays unknown with weight 0 — a missing
row or an empty numeric cell is **never** a silent negative.

In [ ]:
from knee_mri.labels import build_label_table, check_training_readiness, load_reference_table

labels = build_label_table(cfg, study_ids)
counts = labels.counts_frame()
display(counts)

readiness = check_training_readiness(cfg, labels, len(study_ids))
readiness.log()
print("\nready for full training:", readiness.ok)
for reason in readiness.reasons:
    print(" -", reason)

In [ ]:
# The radiologist reference set is loaded separately and never merged into training labels.
reference = load_reference_table(cfg, study_ids)
if reference is not None:
    covered = [s for s, w in zip(reference.study_ids, reference.weights.sum(axis=1)) if w > 0]
    print(f"reference covers {len(covered)} studies —", reference.policy["usage"])

## 2. Manifest and series selection

One row per *volume candidate*: mixed echoes / time points / orientations inside a series
become separate candidates. Slices are ordered by projected `ImagePositionPatient`.

The full scan takes a while — start with a small `limit` here, and run the full build from a
terminal.

In [ ]:
from knee_mri.manifest import build_manifest, load_manifest, select_series, patient_group_audit

SAMPLE = True   # set to False once the full manifest exists

if SAMPLE:
    manifest = build_manifest(cfg, limit=40, out_path=Path(cfg.paths.work_dir) / "manifest_sample" / "manifest.csv")
else:
    manifest = load_manifest(cfg)

manifest[[
    "plane", "plane_angle_deg", "n_slices", "rows", "cols", "row_spacing_mm",
    "slice_spacing_mm", "inplane_ops", "series_description", "echo_time",
    "quality_flags", "usable",
]].head(10)

In [ ]:
series_meta = read_id_csv(cfg.paths.train_series_csv)
selection = select_series(
    cfg, manifest, series_meta,
    out_path=Path(cfg.paths.work_dir) / ("manifest_sample" if SAMPLE else "manifest") / "series_selection.csv",
)
selection[["slot", "selected", "score", "n_candidates", "series_description", "n_slices", "reason"]].head(9)

In [ ]:
# PatientID is audited before it is trusted as a grouping key.
audit = patient_group_audit(manifest, int(cfg.split.max_studies_per_patient_id))
print(audit[["patient_id_missing", "patient_id_inconsistent", "patient_id_site_reused"]].sum().to_dict())
audit.head(3)

## 3. Deterministic preprocessing and cache

Whole preprocessed *series* are cached, before any stochastic centre sampling, so each epoch
can draw a different bag from the same pixels. The cache directory name contains the
preprocessing hash: changing `image_size`, `fov_mm`, the crop policy etc. creates a new cache
rather than silently reusing a stale one.

In [ ]:
from knee_mri.preprocess import build_cache, cache_root, preprocess_signature

print("preprocessing signature:", preprocess_signature(cfg))
print("cache root:", cache_root(cfg))

cache_report = build_cache(cfg, selection, workers=4)
cache_report["status"].value_counts()

## 4. QC gallery — what the model actually receives

Original vs processed views, the physical crop boundary, first/middle/last slices, the real
triplets, and the quality flags. The selection deliberately includes awkward cases.

In [ ]:
from knee_mri.qc import build_qc_gallery, numeric_qc

qc = build_qc_gallery(cfg, selection, manifest, n_studies=6)
print({k: v for k, v in qc.items() if k not in ("details", "studies", "figures")})

In [ ]:
from IPython.display import Image, display as show

for figure in qc["figures"][:3]:
    show(Image(filename=figure))

In [ ]:
# Numeric checks work even when no image can be rendered.
numeric_qc(cfg, qc["studies"])

## 5. Immutable folds

Group-disjoint folds with a multilabel-aware balance. The radiologist reference studies and
their groups are held out of training by default.

In [ ]:
from knee_mri.splits import make_splits, fold_study_ids, assert_group_disjoint

reference_ids = None
if reference is not None:
    reference_ids = [s for s, w in zip(reference.study_ids, reference.weights.sum(axis=1)) if w > 0]

splits = make_splits(
    cfg,
    study_ids=study_ids,
    label_table=labels,
    patient_audit=audit,
    train_df=train_df,
    reference_study_ids=reference_ids,
)
display(splits.groupby(["role", "fold"]).size().rename("n_studies").reset_index())

train_ids, val_ids = fold_study_ids(splits, int(cfg.split.fold))
assert_group_disjoint(splits, int(cfg.split.fold))
print(f"fold {cfg.split.fold}: {len(train_ids)} train / {len(val_ids)} validation studies (group-disjoint)")

## 6. The tiny checks

`selftest` exercises the real model and gradient path on synthetic data: nothing being
checked is mocked away.

In [ ]:
from knee_mri.selftest import run_all_checks

results = run_all_checks(cfg)
pd.DataFrame(results, columns=["check", "passed", "detail"])

### The 2.5D bag contract

`images [P, S, K, H, W]`, `slice_valid_mask [P, S]`, `series_present_mask [P]`,
`targets [12]`, `label_weights [12]` — plus ids and metadata kept out of the tensor path.

In [ ]:
from knee_mri.dataset import StudyBagDataset, collate_studies

cached_ids = [s for s in val_ids if (Path(cache_root(cfg)) / s).exists()][:4]
if cached_ids:
    dataset = StudyBagDataset(cfg, cached_ids, labels, train=False)
    item = dataset[0]
    print("images           ", tuple(item["images"].shape))
    print("slice_valid_mask ", tuple(item["slice_valid_mask"].shape), "valid:", int(item["slice_valid_mask"].sum()))
    print("series_present   ", item["series_present_mask"].tolist())
    print("targets finite   ", bool(item["targets"].isfinite().all()))
    print("supervised targets", int((item["label_weights"] > 0).sum()), "/ 12")
else:
    print("No cached study from this fold yet — build the cache for these studies first.")

In [ ]:
# Synthetic smoke test of the whole training path (no real image needed).
from knee_mri.train import run_synthetic

smoke_cfg = load_config(overrides=[
    "data.image_size=64", "data.centers_per_series=6",
    "train.microbatch_studies=2", "train.accumulation_steps=2",
])
smoke_summary = run_synthetic(smoke_cfg, n_studies=12, epochs=3)
smoke_summary

## 7. One fold

This is the real run. It refuses to start while the label-readiness gate fails — that is the
intended behaviour, not a bug. Use the in-sample overfit diagnostic while coverage is small,
and never report its numbers as held-out performance.

In [ ]:
from knee_mri.train import run_fold, run_overfit

if readiness.ok:
    summary = run_fold(cfg)          # writes history.csv, metrics_per_class.csv, best.pt, ...
else:
    print("Readiness gate failed — running the in-sample overfit diagnostic instead.\n")
    summary = run_overfit(cfg, n_studies=8)

summary

In [ ]:
run_dir = Path(summary["run_dir"])
display(pd.read_csv(run_dir / "history.csv"))
metrics_path = run_dir / "metrics_per_class.csv"
if metrics_path.exists():
    display(pd.read_csv(metrics_path))

## 8. Report, checkpoint evaluation, OOF

`merge-oof` refuses anything less than complete, verified-disjoint fold coverage.

In [ ]:
from knee_mri.report import build_html_report

report_path = build_html_report(run_dir, qc_dir=Path(cfg.paths.work_dir) / "qc", label_counts=counts)
print(report_path)

In [ ]:
from knee_mri.evaluate import evaluate_checkpoint

# evaluate_checkpoint(cfg, run_dir / "best.pt")                                 # held-out fold
# evaluate_checkpoint(cfg, run_dir / "best.pt", partition="reference_holdout")  # diagnostic audit only

In [ ]:
from knee_mri.evaluate import merge_oof

# Only after every intended fold has finished:
# merge_oof(cfg, [
#     "work/runs/fold0_.../validation_predictions.csv",
#     "work/runs/fold1_.../validation_predictions.csv",
#     "work/runs/fold2_.../validation_predictions.csv",
# ])